# Predicting Electric Vehicle Purchases — Exploratory Data Analysis

Reproduces the EDA from the plan-mode analysis. Run this once to sanity-check the dataset before training.

**What this notebook verifies:**
1. Row counts (train: 668,665; test: 286,571).
2. No missing values in either split.
3. No duplicate ids.
4. Class distribution (17.5% positive — `Will_Buy_EV == Yes`).
5. Categorical levels are consistent between train and test.
6. Per-feature signal strength (mean target rate per category / per numeric bin).

**Runtime:** ~30 seconds.

**How to run:**
- **Local:** `jupyter lab` from the project root, open this file, run all.
- **Kaggle:** upload as a notebook, attach the competition dataset, adjust the paths in Cell 1.

In [ ]:
# Cell 1: Imports and path setup
import os
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # non-interactive backend for headless runs
import matplotlib.pyplot as plt

# On Kaggle, the competition data lives at /kaggle/input/playground-series-s6e9/.
if os.path.isdir("/kaggle/input/playground-series-s6e9"):
    DATA_DIR = Path("/kaggle/input/playground-series-s6e9")
else:
    DATA_DIR = Path("data")

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
SAMPLE_CSV = DATA_DIR / "sample_submission.csv"
print(f"Train:      {TRAIN_CSV}  ({TRAIN_CSV.stat().st_size / 1e6:.1f} MB)")
print(f"Test:       {TEST_CSV}  ({TEST_CSV.stat().st_size / 1e6:.1f} MB)")
print(f"Sample sub: {SAMPLE_CSV}  ({SAMPLE_CSV.stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Cell 2: Load raw data
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_CSV)

print(f"train shape: {train.shape}")
print(f"test shape:  {test.shape}")
print(f"sample sub shape: {sample_sub.shape}")
print()
print(f"train columns: {list(train.columns)}")
print(f"test columns:  {list(test.columns)}")
print()
print(f"train dtypes:\n{train.dtypes.to_string()}")

In [ ]:
# Cell 3: Basic integrity checks
print("=== Integrity checks ===")
print(f"Train id unique: {train['id'].is_unique}")
print(f"Test id unique:  {test['id'].is_unique}")
print(f"Train/test id overlap: {len(set(train['id']) & set(test['id']))}")
print()
print(f"Train missing values: {train.isnull().sum().sum()}")
print(f"Test missing values:  {test.isnull().sum().sum()}")
print()
print(f"Train full-row duplicates: {train.drop(columns=['id']).duplicated().sum()}")
print()
print("Class distribution (train):")
print(train["Will_Buy_EV"].value_counts())
print(f"\nPositive rate: {train['Will_Buy_EV'].value_counts(normalize=True)['Yes']:.4f}")
print(f"\nSample sub target distribution: all={sample_sub['Will_Buy_EV'].nunique()} unique values")
print(f"  Will_Buy_EV value: {sample_sub['Will_Buy_EV'].iloc[0]}")

In [ ]:
# Cell 4: Categorical level consistency between train and test
CAT_COLS = [
        "Gender", "City_Type", "Current_Car_Type",
        "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level",
    ]
print("=== Categorical level consistency (train == test) ===")
for col in CAT_COLS:
    train_levels = sorted(train[col].dropna().unique().tolist())
    test_levels = sorted(test[col].dropna().unique().tolist())
    match = train_levels == test_levels
    flag = "OK" if match else "MISMATCH"
    print(f"  {col:30s} {flag:10s} train={train_levels} test={test_levels}")

In [ ]:
# Cell 5: Numeric feature distributions
NUM_COLS = [
    "Age", "Annual_Income_USD", "Daily_Commute_km",
    "Number_of_Cars_Owned", "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work", "Environmental_Concern_Level",
]
print("=== Numeric feature summary (train) ===")
print(train[NUM_COLS].describe().T.to_string())
print()
print("=== Numeric feature summary (test) ===")
print(test[NUM_COLS].describe().T.to_string())

In [ ]:
# Cell 6: Per-feature signal strength (categorical)
print("=== Per-feature target rate (categorical) ===")
for col in CAT_COLS:
    print(f"\n{col}:")
    ct = pd.crosstab(train[col], train["Will_Buy_EV"], normalize="index")
    print((ct * 100).round(2).to_string())

In [ ]:
# Cell 7: Per-feature signal strength (numeric, by target)
print("=== Numeric feature means by target class ===")
summary = train.groupby("Will_Buy_EV")[NUM_COLS].mean().T
summary.columns = ["No_mean", "Yes_mean"]
summary["diff_pct"] = ((summary["Yes_mean"] - summary["No_mean"]) / summary["No_mean"] * 100).round(2)
print(summary.to_string())

In [ ]:
# Cell 8: Quick visualization of class distributions for the strongest signals
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Range_Anxiety_Level vs target rate
ax = axes[0, 0]
ct = pd.crosstab(train["Range_Anxiety_Level"], train["Will_Buy_EV"], normalize="index")
ct["Yes"].plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Will_Buy_EV=Yes rate by Range_Anxiety_Level")
ax.set_ylabel("Yes rate")
ax.set_ylim(0, 0.25)

# 2. Subsidy_Available vs target rate
ax = axes[0, 1]
ct = pd.crosstab(train["Subsidy_Available"], train["Will_Buy_EV"], normalize="index")
ct["Yes"].plot(kind="bar", ax=ax, color="#55A868")
ax.set_title("Will_Buy_EV=Yes rate by Subsidy_Available")
ax.set_ylabel("Yes rate")
ax.set_ylim(0, 0.30)

# 3. Environmental_Concern_Level distribution
ax = axes[1, 0]
train["Environmental_Concern_Level"].hist(bins=5, ax=ax, color="#C44E52", alpha=0.7, label="All")
train.loc[train["Will_Buy_EV"] == "Yes", "Environmental_Concern_Level"].hist(
    bins=5, ax=ax, color="#4C72B0", alpha=0.7, label="Yes only"
)
ax.set_title("Environmental_Concern_Level distribution")
ax.legend()

# 4. Annual_Income_USD distribution by target
ax = axes[1, 1]
train.loc[train["Will_Buy_EV"] == "No", "Annual_Income_USD"].hist(
    bins=50, ax=ax, color="#C44E52", alpha=0.5, label="No"
)
train.loc[train["Will_Buy_EV"] == "Yes", "Annual_Income_USD"].hist(
    bins=50, ax=ax, color="#4C72B0", alpha=0.5, label="Yes"
)
ax.set_title("Annual_Income_USD distribution by target")
ax.legend()

plt.tight_layout()
out_path = Path("docs/eda_figures.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, dpi=100, bbox_inches="tight")
print(f"Saved figure to {out_path}")
plt.close()

## Summary of EDA findings (also in `docs/data_doc.md`)

**Strong signals (large separation between Yes and No):**
1. `Range_Anxiety_Level`: High → 0.14% Yes, Low → 18.9% Yes.
2. `Subsidy_Available`: No → 0.6% Yes, Yes → 27.5% Yes.
3. `Environmental_Concern_Level`: Yes mean = 4.38, No mean = 2.63.
4. `Annual_Income_USD`: Yes mean = $98,827, No mean = $81,795.

**Weak signals:**
- `Home_Charging_Possible`, `City_Type`, `Current_Car_Type`, `Gender`, commute distance, age, number of cars owned.

**Conclusion:** A tree-based model with native categorical handling should easily hit AUC > 0.90 on this dataset. The strong signals are orthogonal (subsidy + anxiety + income), so a single LGBM is likely sufficient for the v1 submission.